# Link Prediction avec GNN — v2
Améliorations :
- **Features structurelles** par paire (degrés, voisins communs, Jaccard, Adamic-Adar, Preferential Attachment, composante connexe, similarité cosinus des features brutes)
- **Soumission en probas** (et non en 0/1) — indispensable si Kaggle évalue en AUC
- **Régularisation** renforcée (dropout + weight_decay) pour réduire l'écart train/val

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement des données

In [2]:
train = pd.read_csv("data/train.txt", sep=" ", header=None)
train.columns = ["u", "v", "label"]

test = pd.read_csv("data/test.txt", sep=" ", header=None)
test.columns = ["u", "v"]

node_info = pd.read_csv("data/node_information.csv", header=None)
node_info = node_info.rename(columns={0: "node"})
node_features = {
    int(row["node"]): row.drop("node").values.astype(np.float32)
    for _, row in node_info.iterrows()
}
feature_dim = len(next(iter(node_features.values())))

# Graphe de message-passing : uniquement les liens positifs
G = nx.Graph()
pos_edges = train[train["label"] == 1][["u", "v"]].values
G.add_edges_from(pos_edges)

print(f"Noeuds : {G.number_of_nodes()} | Arêtes : {G.number_of_edges()}")
print(f"Dimension des features : {feature_dim}")
print(f"Paires train : {len(train)} | Paires test : {len(test)}")

Noeuds : 3597 | Arêtes : 5248
Dimension des features : 932
Paires train : 10496 | Paires test : 3498


## 2. Construction du graphe PyG

In [3]:
all_nodes = sorted(G.nodes())
node_to_idx = {n: i for i, n in enumerate(all_nodes)}
num_nodes = len(all_nodes)

x = torch.zeros((num_nodes, feature_dim), dtype=torch.float)
for node, idx in node_to_idx.items():
    if node in node_features:
        x[idx] = torch.tensor(node_features[node])

edge_list = [
    (node_to_idx[u], node_to_idx[v])
    for u, v in G.edges()
    if u in node_to_idx and v in node_to_idx
]
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

data = Data(x=x, edge_index=edge_index, num_nodes=num_nodes)
print(data)

Data(x=[3597, 932], edge_index=[2, 10496], num_nodes=3597)


## 3. Features structurelles par paire

Ces 8 features sont calculées **directement sur le graphe** (pas d'apprentissage) :
elles capturent des signaux très forts que le GNN ne voit pas nécessairement.

| Feature | Intuition |
|---|---|
| `deg_u`, `deg_v` | Nœuds très connectés → plus de chances d'avoir un lien |
| `common_neighbors` | Amis en commun — signal classique |
| `jaccard` | Overlap normalisé des voisinages |
| `adamic_adar` | Voisins communs peu connectés = signal plus fort |
| `pref_attach` | deg_u × deg_v — baseline très simple mais efficace |
| `same_component` | Liens inter-composantes très improbables |
| `cosine_sim` | Similarité des features brutes de nœuds |

In [4]:
# Pré-calculs pour accélérer
degree = dict(G.degree())
components = {n: c for c, comp in enumerate(nx.connected_components(G)) for n in comp}

# Norme L2 des features (pour cosinus)
feat_norms = {
    node: np.linalg.norm(feat) + 1e-9
    for node, feat in node_features.items()
}


def structural_features(u, v):
    """Retourne un vecteur numpy de 8 features structurelles pour la paire (u, v)."""
    # Degrés (0 si nœud absent du graphe)
    deg_u = degree.get(u, 0)
    deg_v = degree.get(v, 0)

    # Voisinage
    if G.has_node(u) and G.has_node(v):
        nu = set(G.neighbors(u))
        nv = set(G.neighbors(v))
        inter = nu & nv
        union = nu | nv
        cn    = len(inter)
        jacc  = cn / len(union) if union else 0.0
        aa    = sum(1.0 / np.log(degree[w] + 1e-9) for w in inter if degree.get(w, 0) > 1)
        pa    = deg_u * deg_v
        same_comp = float(components.get(u, -1) == components.get(v, -2))
    else:
        cn, jacc, aa, pa, same_comp = 0, 0.0, 0.0, 0, 0.0

    # Similarité cosinus des features brutes
    if u in node_features and v in node_features:
        cosine = np.dot(node_features[u], node_features[v]) / (feat_norms[u] * feat_norms[v])
    else:
        cosine = 0.0

    return np.array([deg_u, deg_v, cn, jacc, aa, pa, same_comp, cosine], dtype=np.float32)


# Calcul sur toutes les paires (train + test)
print("Calcul des features structurelles train...")
train_struct = np.stack([structural_features(r.u, r.v) for r in train.itertuples()])
print("Calcul des features structurelles test...")
test_struct  = np.stack([structural_features(r.u, r.v) for r in test.itertuples()])

# Normalisation (important pour les features à échelle variable : degré, PA)
scaler = StandardScaler()
train_struct = scaler.fit_transform(train_struct).astype(np.float32)
test_struct  = scaler.transform(test_struct).astype(np.float32)

struct_dim = train_struct.shape[1]
print(f"Dimension structurelle : {struct_dim}")
print(f"Exemple paire 0 : {train_struct[0]}")

Calcul des features structurelles train...
Calcul des features structurelles test...
Dimension structurelle : 8
Exemple paire 0 : [-0.36074916  5.293295   -0.16636163 -0.14546403 -0.14235507  1.1185468
  0.         -0.8426868 ]


## 4. Split train / validation

In [5]:
def pairs_to_tensors(df_pairs, struct_arr, node_to_idx):
    """Convertit un DataFrame de paires en tenseurs (indices + features struct + labels)."""
    mask = df_pairs["u"].isin(node_to_idx) & df_pairs["v"].isin(node_to_idx)
    valid_idx = np.where(mask.values)[0]
    valid     = df_pairs.iloc[valid_idx]
    u_idx  = torch.tensor([node_to_idx[u] for u in valid["u"]], dtype=torch.long)
    v_idx  = torch.tensor([node_to_idx[v] for v in valid["v"]], dtype=torch.long)
    struct = torch.tensor(struct_arr[valid_idx], dtype=torch.float)
    labels = torch.tensor(valid["label"].values, dtype=torch.float)
    return u_idx, v_idx, struct, labels


idx_all = np.arange(len(train))
tr_idx, va_idx = train_test_split(idx_all, test_size=0.2, stratify=train["label"], random_state=42)

train_u, train_v, train_s, train_y = pairs_to_tensors(train.iloc[tr_idx], train_struct[tr_idx], node_to_idx)
val_u,   val_v,   val_s,   val_y   = pairs_to_tensors(train.iloc[va_idx], train_struct[va_idx], node_to_idx)

print(f"Train : {len(train_y)} | Val : {len(val_y)}")
print(f"Ratio positifs (train) : {train_y.mean():.2%}")

Train : 8396 | Val : 2100
Ratio positifs (train) : 50.00%


## 5. Modèles

Architecture : **GNN encoder** → embeddings `z_u`, `z_v`  
→ concaténation avec les **features structurelles** de la paire  
→ **LinkPredictor** MLP → logit → sigmoid → probabilité

In [6]:
class GNN_model(torch.nn.Module):
    """Encodeur GCN avec BatchNorm pour stabiliser l'entraînement."""
    def __init__(self, num_layers, input_size, hidden_size, output_size, dropout=0.4):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns   = torch.nn.ModuleList()
        self.convs.append(GCNConv(input_size, hidden_size))
        self.bns.append(torch.nn.BatchNorm1d(hidden_size))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_size, hidden_size))
            self.bns.append(torch.nn.BatchNorm1d(hidden_size))
        self.convs.append(GCNConv(hidden_size, output_size))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs[:-1], self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x


class LinkPredictor(torch.nn.Module):
    """
    MLP qui combine :
    - les embeddings GNN de u et v  [z_u | z_v]
    - les features structurelles de la paire
    """
    def __init__(self, emb_dim, struct_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        in_dim = emb_dim * 2 + struct_dim
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim),
            torch.nn.BatchNorm1d(hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim, hidden_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, z_u, z_v, struct_feats):
        """Retourne un logit (avant sigmoid) pour chaque paire."""
        x = torch.cat([z_u, z_v, struct_feats], dim=-1)
        return self.net(x).squeeze(-1)

## 6. Entraînement

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

NUM_LAYERS   = 3
HIDDEN_SIZE  = 128
EMB_SIZE     = 64
DROPOUT_GNN  = 0.4
DROPOUT_MLP  = 0.3
LR           = 1e-3
WEIGHT_DECAY = 1e-4   # L2 régularisation
EPOCHS       = 500
PATIENCE     = 50     # early stopping

# Tout sur device
data    = data.to(device)
train_u, train_v, train_s, train_y = (
    train_u.to(device), train_v.to(device), train_s.to(device), train_y.to(device)
)
val_u, val_v, val_s, val_y = (
    val_u.to(device), val_v.to(device), val_s.to(device), val_y.to(device)
)

gnn       = GNN_model(NUM_LAYERS, feature_dim, HIDDEN_SIZE, EMB_SIZE, DROPOUT_GNN).to(device)
predictor = LinkPredictor(EMB_SIZE, struct_dim, HIDDEN_SIZE, DROPOUT_MLP).to(device)
optimizer = torch.optim.Adam(
    list(gnn.parameters()) + list(predictor.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=20, verbose=True
)
pos_weight = torch.tensor([(train_y == 0).sum() / (train_y == 1).sum()]).to(device)


def evaluate(z, u_idx, v_idx, s, y):
    with torch.no_grad():
        logits = predictor(z[u_idx], z[v_idx], s)
        loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
        probs  = torch.sigmoid(logits).cpu().numpy()
        auc    = roc_auc_score(y.cpu().numpy(), probs)
    return loss.item(), auc


best_val_auc  = 0.0
best_state    = None
epochs_no_imp = 0

for epoch in range(1, EPOCHS + 1):
    gnn.train(); predictor.train()
    optimizer.zero_grad()

    z      = gnn(data.x, data.edge_index)
    logits = predictor(z[train_u], z[train_v], train_s)
    loss   = F.binary_cross_entropy_with_logits(logits, train_y, pos_weight=pos_weight)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        gnn.eval(); predictor.eval()
        z_eval = gnn(data.x, data.edge_index)
        tr_loss, tr_auc = evaluate(z_eval, train_u, train_v, train_s, train_y)
        va_loss, va_auc = evaluate(z_eval, val_u,   val_v,   val_s,   val_y)
        scheduler.step(va_auc)
        print(
            f"Epoch {epoch:>3} | "
            f"Train loss {tr_loss:.4f} AUC {tr_auc:.4f} | "
            f"Val   loss {va_loss:.4f} AUC {va_auc:.4f}"
        )
        if va_auc > best_val_auc:
            best_val_auc  = va_auc
            epochs_no_imp = 0
            best_state = {
                "gnn":       {k: v.cpu() for k, v in gnn.state_dict().items()},
                "predictor": {k: v.cpu() for k, v in predictor.state_dict().items()},
            }
        else:
            epochs_no_imp += 20
            if epochs_no_imp >= PATIENCE:
                print(f"Early stopping à l'epoch {epoch}.")
                break

print(f"\nMeilleur AUC validation : {best_val_auc:.4f}")

Device : cpu


d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch  20 | Train loss 0.6194 AUC 0.8621 | Val   loss 0.6260 AUC 0.8438
Epoch  40 | Train loss 0.4510 AUC 0.9071 | Val   loss 0.5191 AUC 0.8722
Epoch  60 | Train loss 0.1306 AUC 0.9903 | Val   loss 0.3069 AUC 0.9453
Epoch  80 | Train loss 0.0368 AUC 0.9990 | Val   loss 0.2243 AUC 0.9713
Epoch 100 | Train loss 0.0147 AUC 0.9996 | Val   loss 0.2197 AUC 0.9739
Epoch 120 | Train loss 0.0082 AUC 1.0000 | Val   loss 0.2232 AUC 0.9751
Epoch 140 | Train loss 0.0039 AUC 1.0000 | Val   loss 0.2351 AUC 0.9749
Epoch 160 | Train loss 0.0027 AUC 1.0000 | Val   loss 0.2341 AUC 0.9763
Epoch 180 | Train loss 0.0015 AUC 1.0000 | Val   loss 0.2416 AUC 0.9760
Epoch 200 | Train loss 0.0017 AUC 1.0000 | Val   loss 0.2346 AUC 0.9772
Epoch 220 | Train loss 0.0012 AUC 1.0000 | Val   loss 0.2548 AUC 0.9756
Epoch 240 | Train loss 0.0009 AUC 1.0000 | Val   loss 0.2502 AUC 0.9770
Epoch 260 | Train loss 0.0007 AUC 1.0000 | Val   loss 0.2607 AUC 0.9762
Early stopping à l'epoch 260.

Meilleur AUC validation : 0.9772


## 7. Inférence et soumission

> **Important** : Kaggle évalue en AUC → on soumet les **probabilités** (`score`), pas les labels binaires.  
> Binariser revient à perdre toute l'information de ranking et fait chuter l'AUC.

In [8]:
# Rechargement du meilleur checkpoint
gnn.load_state_dict({k: v.to(device) for k, v in best_state["gnn"].items()})
predictor.load_state_dict({k: v.to(device) for k, v in best_state["predictor"].items()})
gnn.eval(); predictor.eval()

with torch.no_grad():
    z = gnn(data.x, data.edge_index)

# Tenseurs test
test_u_idx, test_v_idx, test_probs_list = [], [], []
test_s_tensor = torch.tensor(test_struct, dtype=torch.float, device=device)

scores = []
BATCH  = 512  # évite les OOM sur GPU
n_test = len(test)

# Pré-indexation
u_list = [node_to_idx.get(int(r.u), -1) for r in test.itertuples()]
v_list = [node_to_idx.get(int(r.v), -1) for r in test.itertuples()]

with torch.no_grad():
    for start in range(0, n_test, BATCH):
        end  = min(start + BATCH, n_test)
        us   = u_list[start:end]
        vs   = v_list[start:end]
        s_b  = test_s_tensor[start:end]

        batch_probs = []
        for i, (ui, vi) in enumerate(zip(us, vs)):
            if ui == -1 or vi == -1:
                batch_probs.append(0.1)  # nœud hors graphe → proba neutre basse
            else:
                logit = predictor(
                    z[ui].unsqueeze(0),
                    z[vi].unsqueeze(0),
                    s_b[i].unsqueeze(0)
                )
                batch_probs.append(torch.sigmoid(logit).item())
        scores.extend(batch_probs)

# ─── Soumission : PROBAS, pas des 0/1 ───────────────────────────────────────
test["score"] = scores

submit = pd.DataFrame({
    "ID"       : range(len(scores)),
    "Predicted": scores,          # <-- probas continues
})
submit.to_csv("GNN_predictions_strucfeats.csv", index=False)

print(test[["u", "v", "score"]].head(10).to_string(index=False))
print(f"\nScore min : {min(scores):.4f}  max : {max(scores):.4f}  mean : {np.mean(scores):.4f}")
print(f"Paires > 0.5 : {(np.array(scores) > 0.5).sum()} / {n_test}")

   u    v        score
3425 4524 7.315438e-04
1620 2617 9.820608e-01
4832 6317 3.798249e-06
4984 7298 3.552697e-04
 385 5481 1.689709e-03
1722 2930 8.095451e-07
1534 3330 1.013341e-01
5015 6354 1.707966e-05
 856 2504 6.001799e-04
 851 5515 1.323559e-01

Score min : 0.0000  max : 1.0000  mean : 0.2677
Paires > 0.5 : 912 / 3498
